In [0]:
from pyspark.sql import functions as F

staging_order_items = spark.table("brazilian_ecommerce.prod.staging_order_items")
staging_products = spark.table("brazilian_ecommerce.prod.staging_products")
staging_category_translation = spark.table("brazilian_ecommerce.prod.staging_category_translation")
staging_order_reviews = spark.table("brazilian_ecommerce.prod.staging_order_reviews")
staging_orders = spark.table("brazilian_ecommerce.prod.staging_orders")

# --- overall sales metrics: all orders, all items (no shared-blame issue for revenue/units) ---
product_sales = (
    staging_order_items
    .groupBy("product_id")
    .agg(
        F.count("order_item_id").alias("total_units_sold"),
        F.sum("price").cast("decimal(10,2)").alias("total_revenue"),
    )
)

# --- identify orders with exactly ONE DISTINCT product ---
single_product_orders = (
    staging_order_items
    .groupBy("order_id")
    .agg(
        F.countDistinct("product_id").alias("distinct_product_count"),
        F.first("product_id").alias("product_id"),   # safe: only 1 distinct product per order after filter
    )
    .filter(F.col("distinct_product_count") == 1)
    .select("order_id", "product_id")
)

# --- avg review score: single-product orders only ---
product_review_scores = (
    single_product_orders
    .join(staging_order_reviews.select("order_id", "review_score"), on="order_id", how="left")
    .groupBy("product_id")
    .agg(F.avg("review_score").cast("decimal(3,2)").alias("avg_review_score"))
)

# --- avg delivery days: single-product orders only ---
product_delivery_days = (
    single_product_orders
    .join(
        staging_orders.select("order_id", "order_purchase_timestamp", "order_delivered_customer_date"),
        on="order_id",
        how="left",
    )
    .withColumn(
        "delivery_days",
        F.datediff(F.to_date("order_delivered_customer_date"), F.to_date("order_purchase_timestamp"))
    )
    .groupBy("product_id")
    .agg(F.avg("delivery_days").alias("avg_delivery_days"))
)

# --- combine everything ---
gold_product_performance = (
    product_sales
    .join(product_review_scores, on="product_id", how="left")
    .join(product_delivery_days, on="product_id", how="left")
    .join(
        staging_products.select("product_id", "product_category_name"),
        on="product_id",
        how="left",
    )
    .join(staging_category_translation, on="product_category_name", how="left")
    .withColumn(
        "product_category_name_english",
        F.coalesce(
            F.col("product_category_name_english"),
            F.col("product_category_name"),
            F.lit("uncategorized"),
        )
    )
    .select(
        "product_id",
        "product_category_name_english",
        "total_units_sold",
        "total_revenue",
        "avg_review_score",
        "avg_delivery_days",
    )
    .withColumn("_loaded_at", F.current_timestamp())
    .orderBy(F.desc("total_revenue"))
)

full_table_name = "brazilian_ecommerce.prod.gold_product_performance"

(
    gold_product_performance.write
    .mode("overwrite")
    .option("comment", "gold table: one row per product, with sales volume/revenue (all orders) and review/delivery metrics restricted to single-product orders")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ALTER COLUMN product_id SET NOT NULL
""")

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ADD CONSTRAINT pk_gold_product_performance PRIMARY KEY (product_id)
""")

print(f"Loaded {gold_product_performance.count()} rows into {full_table_name}")